## 1. Imports & Configuration


In [ ]:
from pathlib import Path
import random
import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import confusion_matrix

# ================= Config =================
SEED = 42
DATA_DIR = Path("dataset") # فولدر اصلی دیتاست

IMAGE_SIZE = 64
BATCH_SIZE = 32

VAL_RATIO = 0.10 # 10 درصد از دیتای Train برای Validation جدا می شود

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


## 2. Dataset Utilities & Splitting


In [ ]:
def scan_image_folder(data_dir):
    data_dir = Path(data_dir)
    if not data_dir.exists():
        raise FileNotFoundError(f"Folder not found: {data_dir}")

    rows = []
    class_dirs = sorted([p for p in data_dir.iterdir() if p.is_dir()])
    if not class_dirs:
        raise ValueError(f"No class folders found inside {data_dir}")

    for class_dir in class_dirs:
        label = class_dir.name
        for image_path in class_dir.rglob("*"):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                rows.append({"path": str(image_path.as_posix()), "label": label})

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f"No images found in {data_dir}")

    return df.sample(frac=1, random_state=SEED).reset_index(drop=True)

def split_train_val(df, val_ratio=VAL_RATIO):
    labels = df["label"]
    can_stratify = labels.value_counts().min() >= 2
    
    train_df, val_df = train_test_split(
        df, 
        test_size=val_ratio, 
        random_state=SEED, 
        shuffle=True, 
        stratify=labels if can_stratify else None
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)

def get_class_distribution(df):
    return df["label"].value_counts().rename_axis("label").reset_index(name="count").sort_values("label").reset_index(drop=True)

def sample_images_by_class(df, max_per_class=1):
    samples = []
    rng = random.Random(SEED)
    for label in sorted(df["label"].unique()):
        class_paths = df[df["label"] == label]["path"].tolist()
        rng.shuffle(class_paths)
        for path in class_paths[:max_per_class]:
            samples.append({"label": label, "path": path})
    return pd.DataFrame(samples)


## 3. PyTorch Dataset & DataLoaders


In [ ]:
class SignLanguageDataset(Dataset):
    def __init__(self, dataframe, label_to_index=None, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        if label_to_index is None:
            labels = sorted(self.dataframe["label"].unique())
            self.label_to_index = {label: index for index, label in enumerate(labels)}
        else:
            self.label_to_index = label_to_index

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = Image.open(row["path"]).convert("RGB")
        label = self.label_to_index[row["label"]]
        if self.transform is not None:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

def get_phase1_transforms(image_size=IMAGE_SIZE):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
    ])

def build_dataloaders(train_df, val_df, test_df, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE):
    label_to_index = {label: index for index, label in enumerate(sorted(train_df["label"].unique()))}
    transform = get_phase1_transforms(image_size=image_size)

    train_loader = DataLoader(SignLanguageDataset(train_df, label_to_index, transform), batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(SignLanguageDataset(val_df, label_to_index, transform), batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(SignLanguageDataset(test_df, label_to_index, transform), batch_size=batch_size, shuffle=False, num_workers=0)

    return train_loader, val_loader, test_loader, label_to_index


## 4. Visualization Utilities


In [ ]:
def plot_class_distribution(df, title="Class Distribution"):
    distribution = get_class_distribution(df)
    fig, ax = plt.subplots(figsize=(max(8, len(distribution) * 0.6), 5))
    ax.bar(distribution["label"], distribution["count"])
    ax.set_title(title)
    ax.set_xlabel("Class")
    ax.set_ylabel("Number of Images")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    plt.show()

def plot_sample_grid(df, max_per_class=1):
    samples = sample_images_by_class(df, max_per_class=max_per_class)
    n = len(samples)
    if n == 0: return
    cols = min(5, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = [axes] if rows == 1 and cols == 1 else list(axes) if rows == 1 else axes.flatten()
    
    for ax, (_, row) in zip(axes, samples.iterrows()):
        ax.imshow(Image.open(row["path"]).convert("RGB"))
        ax.set_title(row["label"])
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle("Random Sample from Each Class", y=1.02)
    fig.tight_layout()
    plt.show()


## 5. Model Architecture & Training Loop


In [ ]:
class SignLanguageCNN(nn.Module):
    def __init__(self, num_classes):
        super(SignLanguageCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.nnl1 = nn.Linear(128 * 8 * 8, 512)
        self.dropout = nn.Dropout(0.5)
        self.nnl2 = nn.Linear(512, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = torch.flatten(x, 1)
        x = self.relu(self.nnl1(x))
        x = self.dropout(x)
        x = self.nnl2(x)
        return x

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, device='cuda'):
    model.to(device)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct_train, total_train = 0.0, 0, 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = correct_train / total_train
        
        model.eval()
        val_loss, correct_val, total_val = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                
        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = correct_val / total_val
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)
        
        print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")
              
    return history

def plot_learning_curves(history):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], label='Train Loss')
    plt.plot(epochs, history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], label='Train Accuracy')
    plt.plot(epochs, history['val_acc'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.show()

def evaluate_and_plot_confusion_matrix(model, test_loader, class_names, device='cuda'):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix on Test Set')
    plt.show()


## 6. Main Pipeline Execution


In [ ]:
# 1. تنظیم سخت‌افزار
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# 2. خواندن تصاویر از فولدرهای مجزا
print("Scanning Train and Test folders...")
train_full_df = scan_image_folder(DATA_DIR / "train")
test_df = scan_image_folder(DATA_DIR / "test")
print(f"Total images found in train folder: {len(train_full_df)}")
print(f"Total images found in test folder: {len(test_df)}")

# (اختیاری) نمایش توزیع دیتای آموزشی
plot_class_distribution(train_full_df, title="Train Set Class Distribution")

# 3. جدا کردن Validation از دیتای Train
train_df, val_df = split_train_val(train_full_df, val_ratio=VAL_RATIO)
print(f"Final Data Splits -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# 4. ایجاد DataLoaders
train_loader, val_loader, test_loader, label_to_index = build_dataloaders(
    train_df, val_df, test_df, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)
class_names = list(label_to_index.keys())

# 5. مقداردهی مدل و شروع آموزش
model = SignLanguageCNN(num_classes=len(class_names))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("\nStarting training...")
history = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, device=device)

# 6. ارزیابی نهایی روی دیتای Test اصلی
plot_learning_curves(history)
evaluate_and_plot_confusion_matrix(model, test_loader, class_names, device=device)
